# Capstone: Refresh / Content Opportunity Scoring
Question: **which pages are likely to decline next, so editors can review them first?**
Run top to bottom in Colab. Run Step 2 first, then edit the CONFIG in Step 3 to match the real column names.

## Step 1. Download the release files (kept in `data/`, never committed)

In [ ]:
!pip -q install duckdb scikit-learn matplotlib pandas pyarrow
import os, urllib.request
BASE = "https://huggingface.co/buckets/stanley-metone/internship-warehouse-bucket/resolve/"
FILES = {"clients":"dim_clients.parquet","content":"dim_content.parquet",
         "fact":"fact_content_daily_performance_sample.parquet","query":"fact_content_query_90d.parquet"}
os.makedirs("data", exist_ok=True)
tok = os.environ.get("HF_TOKEN")  # optional: only needed if the bucket asks for auth
for k, f in FILES.items():
    p = f"data/{f}"
    if not os.path.exists(p):
        req = urllib.request.Request(BASE+f+"?download=true", headers={"Authorization":f"Bearer {tok}"} if tok else {})
        with urllib.request.urlopen(req) as r, open(p,"wb") as o: o.write(r.read())
    print(k, p, round(os.path.getsize(p)/1e6,1), "MB")

## Step 2. Discover the schema (send me this output if anything fails later)

In [ ]:
import duckdb, pandas as pd
con = duckdb.connect()
for k, f in FILES.items():
    print("\n==", f)
    print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('data/{f}')").df()[["column_name","column_type"]].to_string(index=False))
    print(con.sql(f"SELECT * FROM read_parquet('data/{f}') LIMIT 3").df().to_string())

## Step 3. CONFIG: map the real column names (edit after reading Step 2)

In [ ]:
CFG = dict(
    client="client_id",       # client key in the fact table
    content="content_id",     # content key in the fact table
    date="report_date",       # daily date column
    clicks="clicks",
    impressions="impressions",
    position="avg_position",  # set to None if there is no position column
)
MIN_CLICKS = 30       # a page needs this many clicks in the feature window to be scored
DECLINE_DROP = 0.20   # label = daily clicks fell by at least 20% in the label window
FEATURE_SHARE = 2/3   # first 2/3 of dates = feature window, last 1/3 = label window
SEED = 42

## Step 4. Build the panel (time-aware: features come only from the past, label from the future)
Pages with no rows in the label window are treated as 0 clicks. This is an assumption; check it in Limitations.

In [ ]:
from datetime import timedelta
c = CFG
con.sql(f"""CREATE OR REPLACE VIEW f AS SELECT {c['client']} AS cl, {c['content']} AS ct,
  CAST({c['date']} AS DATE) AS dt, CAST({c['clicks']} AS DOUBLE) AS clk, CAST({c['impressions']} AS DOUBLE) AS imp,
  {('CAST('+c['position']+' AS DOUBLE)') if c['position'] else 'NULL'} AS pos
  FROM read_parquet('data/{FILES['fact']}')""")
dmin, dmax = con.sql("SELECT min(dt), max(dt) FROM f").fetchone()
span = (dmax - dmin).days + 1
cut = dmin + timedelta(days=int(span*FEATURE_SHARE) - 1)
F_days = (cut - dmin).days + 1; L_days = (dmax - cut).days
print("window:", dmin, "->", dmax, "| feature end:", cut, "| feature days:", F_days, "| label days:", L_days)

panel = con.sql(f"""
WITH feat AS (
  SELECT cl, ct, sum(clk) clicks, sum(imp) imps, count(DISTINCT dt) active_days,
         sum(pos*imp)/NULLIF(sum(CASE WHEN pos IS NOT NULL THEN imp END),0) avg_pos,
         sum(CASE WHEN dt > DATE '{cut}' - 7 THEN clk END) last7,
         sum(CASE WHEN dt <= DATE '{cut}' - 7 AND dt > DATE '{cut}' - 14 THEN clk END) prev7
  FROM f WHERE dt <= DATE '{cut}' GROUP BY 1,2),
lab AS (SELECT cl, ct, sum(clk) lab_clicks FROM f WHERE dt > DATE '{cut}' GROUP BY 1,2)
SELECT feat.*, coalesce(lab.lab_clicks,0) lab_clicks FROM feat LEFT JOIN lab USING (cl, ct)
WHERE feat.clicks >= {MIN_CLICKS}""").df()

panel["ctr"] = panel.clicks / panel.imps.clip(lower=1)
panel["momentum"] = (panel.last7.fillna(0) + 1) / (panel.prev7.fillna(0) + 1)
panel["feat_cpd"] = panel.clicks / F_days
panel["label"] = ((panel.lab_clicks / L_days) <= (1 - DECLINE_DROP) * panel.feat_cpd).astype(int)
print(panel.shape, "| decline rate:", round(panel.label.mean(), 3))

## Step 5. Baseline vs model on the same grouped split
Split by client so no client appears in both train and test. Baseline = "recent momentum is falling" (lowest last-7/prev-7 ratio ranks first).

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
FEATS = ["clicks","imps","active_days","avg_pos","ctr","momentum","last7","prev7"]
X = panel[FEATS].astype(float); X["clicks"] = np.log1p(X.clicks); X["imps"] = np.log1p(X.imps)
y = panel.label.values; g = panel.cl.values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED).split(X, y, g))
model = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.08, random_state=SEED).fit(X.iloc[tr], y[tr])
p_model = model.predict_proba(X.iloc[te])[:, 1]
p_base = -np.log(panel.momentum.values[te])  # falling momentum -> higher risk
def prec_at(y_true, s, frac=0.05):
    k = max(1, int(len(s) * frac)); return y_true[np.argsort(-s)[:k]].mean()
res = pd.DataFrame({m: {"ROC-AUC": roc_auc_score(y[te], s), "PR-AUC": average_precision_score(y[te], s),
                        "Precision@top5%": prec_at(y[te], s)} for m, s in [("Baseline", p_base), ("Model", p_model)]}).T
res["Base rate"] = y[te].mean(); display(res.round(3))

## Step 6. Leakage checks

In [ ]:
assert panel.shape[0] > 0 and set(FEATS).isdisjoint({"lab_clicks","label"}), "label-window columns must not be features"
assert str(con.sql(f"SELECT max(dt) FROM f WHERE dt <= DATE '{cut}'").fetchone()[0]) <= str(cut)
assert set(panel.cl.values[tr]).isdisjoint(set(panel.cl.values[te])), "clients overlap across split"
rng = np.random.default_rng(SEED)
shuf = HistGradientBoostingClassifier(max_depth=4, random_state=SEED).fit(X.iloc[tr], rng.permutation(y[tr]))
print("Shuffled-label AUC (should be ~0.5):", round(roc_auc_score(y[te], shuf.predict_proba(X.iloc[te])[:,1]), 3))
print("All leakage checks passed")

## Step 7. Charts (saved to `docs/figures/` for the paper)

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
os.makedirs("docs/figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(6,4))
for n, s in [("Baseline", p_base), ("Model", p_model)]:
    pr, rc, _ = precision_recall_curve(y[te], s); ax.plot(rc, pr, label=n)
ax.axhline(y[te].mean(), ls="--", c="grey", label="Base rate"); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Decline risk: model vs baseline (held-out clients)"); ax.legend(); fig.tight_layout()
fig.savefig("docs/figures/pr_curve.png", dpi=160)
from sklearn.inspection import permutation_importance
pi = permutation_importance(model, X.iloc[te], y[te], n_repeats=3, random_state=SEED, scoring="average_precision")
s = pd.Series(pi.importances_mean, index=FEATS).sort_values()
fig, ax = plt.subplots(figsize=(6,4)); s.plot.barh(ax=ax); ax.set_title("Feature importance (permutation, PR-AUC drop)")
fig.tight_layout(); fig.savefig("docs/figures/feature_importance.png", dpi=160)

## Step 8. Ranked recommendations with reason codes (aggregates only, no IDs published)

In [ ]:
test = panel.iloc[te].copy(); test["risk"] = p_model
q = test.risk.quantile(0.95)
def codes(r):
    out = []
    if r.momentum < 0.7: out.append("MOMENTUM_DOWN")
    if pd.notna(r.avg_pos) and r.avg_pos > 10: out.append("LOW_RANK_POSITION")
    if r.ctr < test.ctr.median() * 0.7: out.append("CTR_BELOW_MEDIAN")
    if r.active_days < F_days * 0.6: out.append("INTERMITTENT_VISIBILITY")
    return out or ["MODEL_ONLY"]
top = test[test.risk >= q].copy(); top["reasons"] = top.apply(codes, axis=1)
playbook = (top.explode("reasons").groupby("reasons").agg(pages=("cl","size"), avg_risk=("risk","mean"), avg_daily_clicks=("feat_cpd","mean"))
            .sort_values("pages", ascending=False).round(3))
display(playbook)
os.makedirs("outputs", exist_ok=True)
playbook.to_csv("outputs/playbook_summary.csv")   # aggregate table, safe to publish
res.round(4).to_csv("outputs/results.csv")

## Step 9. Limitations (copy into the paper)
- Observed associations, not causal claims; nothing here proves how Google ranks pages.
- The sample file covers one month, so the label is a short-horizon decline. Rerun on several months of the full warehouse for a stronger time-aware design.
- Missing label-window rows are treated as zero clicks.
- Results are decision-support: a ranked review list, not an automatic rewrite instruction.